In [10]:
from datetime import date

import nflreadpy as nfl
import pandas as pd

current_year = date.today().year
seasons = list(range(current_year - 4, current_year + 1))
print("Requested seasons:", seasons)

Requested seasons: [2022, 2023, 2024, 2025, 2026]


In [11]:
def load_stats(load_fn, seasons: list[int]) -> pd.DataFrame:
    """Load stats for as many of the requested seasons as are available."""
    frames: list[pd.DataFrame] = []
    loaded: list[int] = []
    for season in seasons:
        try:
            frame = load_fn(season).to_pandas()
        except Exception as exc:  # noqa: BLE001 - missing seasons are expected
            print(f"Skipping {season}: {exc}")
            continue
        if frame.empty:
            print(f"Skipping {season}: no rows returned")
            continue
        frames.append(frame)
        loaded.append(season)
    if not frames:
        raise RuntimeError("No seasons returned data")
    print("Loaded seasons:", loaded)
    return pd.concat(frames, ignore_index=True)


team_stats = load_stats(nfl.load_team_stats, seasons)
player_stats = load_stats(nfl.load_player_stats, seasons)
print("team_stats shape:", team_stats.shape)
print("player_stats shape (before inactive filter):", player_stats.shape)

players = nfl.load_players().to_pandas()
latest_loaded_season = int(player_stats["season"].max())
eligible_players = players[
    (players["status"] != "RET")
    & (players["last_season"].fillna(0) >= latest_loaded_season)
][["gsis_id"]].dropna()

player_stats = player_stats.merge(
    eligible_players,
    left_on="player_id",
    right_on="gsis_id",
    how="inner",
).drop(columns=["gsis_id"])
print(
    "player_stats shape (after excluding RET / stale last_season):",
    player_stats.shape,
)
print("Eligibility cutoff last_season >=", latest_loaded_season)

Skipping 2026: Failed to download https://github.com/nflverse/nflverse-data/releases/download/stats_team/stats_team_week_2026.parquet: 404 Client Error: Not Found for url: https://github.com/nflverse/nflverse-data/releases/download/stats_team/stats_team_week_2026.parquet
Loaded seasons: [2022, 2023, 2024, 2025]
Skipping 2026: Failed to download https://github.com/nflverse/nflverse-data/releases/download/stats_player/stats_player_week_2026.parquet: 404 Client Error: Not Found for url: https://github.com/nflverse/nflverse-data/releases/download/stats_player/stats_player_week_2026.parquet
Loaded seasons: [2022, 2023, 2024, 2025]
team_stats shape: (2278, 138)
player_stats shape (before inactive filter): (75879, 150)
player_stats shape (after excluding RET / stale last_season): (66432, 150)
Eligibility cutoff last_season >= 2025


In [12]:
team_stats

,season,week,team,season_type,game_id,opponent_team,completions,attempts,passing_yards,passing_tds,...,pt_yards,pt_inside_20,pt_out_of_bounds,pt_downed,pt_touchback,pt_fair_caught,pt_returned,pt_return_yards,pt_return_tds,pt_net_yards
0,2022,1,ARI,REG,2022_01_KC_ARI,KC,24,38,205,2,...,255,0,0,0,0,0,5,44,0,211
1,2022,1,ATL,REG,2022_01_NO_ATL,NO,20,33,215,0,...,180,2,0,0,0,3,1,12,0,168
2,2022,1,BAL,REG,2022_01_BAL_NYJ,NYJ,17,30,213,3,...,291,3,0,0,1,3,2,30,0,241
3,2022,1,BUF,REG,2022_01_BUF_LA,LA,26,31,297,3,...,0,0,0,0,0,0,0,0,0,0
4,2022,1,CAR,REG,2022_01_CLE_CAR,CLE,16,27,235,1,...,250,0,0,0,0,0,5,29,0,221
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2273,2025,21,LA,POST,2025_21_LA_SEA,SEA,22,35,374,3,...,84,1,0,0,0,2,0,0,0,84
2274,2025,21,NE,POST,2025_21_NE_DEN,DEN,10,21,86,0,...,303,1,3,1,0,1,3,8,0,295
2275,2025,21,SEA,POST,2025_21_LA_SEA,LA,25,36,346,3,...,226,5,0,0,0,3,2,0,0,226
2276,2025,22,NE,POST,2025_22_SEA_NE,SEA,27,43,295,2,...,356,4,0,0,0,6,2,16,0,340


In [13]:
player_stats

,player_id,player_name,player_display_name,position,position_group,headshot_url,season,week,season_type,game_id,...,pt_out_of_bounds,pt_downed,pt_touchback,pt_fair_caught,pt_returned,pt_return_yards,pt_return_tds,pt_net_yards,fantasy_points,fantasy_points_ppr
0,00-0023459,A.Rodgers,Aaron Rodgers,QB,QB,https://static.www.nfl.com/image/upload/f_auto...,2022,1,REG,2022_01_GB_MIN,...,0,0,0,0,0,0,0,0,3.70,3.70
1,00-0023853,M.Prater,Matt Prater,K,SPEC,https://static.www.nfl.com/image/upload/f_auto...,2022,1,REG,2022_01_KC_ARI,...,0,0,0,0,0,0,0,0,0.00,0.00
2,00-0025565,N.Folk,Nick Folk,K,SPEC,https://static.www.nfl.com/image/upload/f_auto...,2022,1,REG,2022_01_NE_MIA,...,0,0,0,0,0,0,0,0,0.00,0.00
3,00-0026158,J.Flacco,Joe Flacco,QB,QB,https://static.www.nfl.com/image/upload/f_auto...,2022,1,REG,2022_01_BAL_NYJ,...,0,0,0,0,0,0,0,0,14.36,14.36
4,00-0026190,C.Campbell,Calais Campbell,DE,DL,https://static.www.nfl.com/image/upload/f_auto...,2022,1,REG,2022_01_BAL_NYJ,...,0,0,0,0,0,0,0,0,0.00,0.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
66427,00-0040647,R.Mills,Rylie Mills,DT,DL,https://static.www.nfl.com/image/upload/f_auto...,2025,22,POST,2025_22_SEA_NE,...,0,0,0,0,0,0,0,0,0.00,0.00
66428,00-0040699,W.Campbell,Will Campbell,OT,OL,https://static.www.nfl.com/image/upload/f_auto...,2025,22,POST,2025_22_SEA_NE,...,0,0,0,0,0,0,0,0,0.00,0.00
66429,00-0040716,C.Woodson,Craig Woodson,SAF,DB,https://static.www.nfl.com/image/upload/f_auto...,2025,22,POST,2025_22_SEA_NE,...,0,0,0,0,0,0,0,0,0.00,0.00
66430,00-0040733,N.Emmanwori,Nick Emmanwori,SAF,DB,https://static.www.nfl.com/image/upload/f_auto...,2025,22,POST,2025_22_SEA_NE,...,0,0,0,0,0,0,0,0,0.00,0.00
